### Quickstart: Compare runs, choose a model, and deploy it to a REST API

In this quickstart, you will:

- Run a hyperparameter sweep on a training script

- Compare the results of the runs in the MLflow UI

- Choose the best run and register it as a model

- Deploy the model to a REST API

- Build a container image suitable for deployment to a cloud platform


In [1]:
import keras
import numpy as np
import pandas as pd 
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.models import infer_signature
import optuna
from optuna import create_study, load_study

d:\Interview\MLOPS\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## load the dataset

data = pd.read_csv("https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",)

data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [38]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4898 entries, 0 to 4897
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         4898 non-null   float64
 1   volatile acidity      4898 non-null   float64
 2   citric acid           4898 non-null   float64
 3   residual sugar        4898 non-null   float64
 4   chlorides             4898 non-null   float64
 5   free sulfur dioxide   4898 non-null   float64
 6   total sulfur dioxide  4898 non-null   float64
 7   density               4898 non-null   float64
 8   pH                    4898 non-null   float64
 9   sulphates             4898 non-null   float64
 10  alcohol               4898 non-null   float64
 11  quality               4898 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 459.3 KB


In [40]:
data["quality"].value_counts()

quality
6    2198
5    1457
7     880
8     175
4     163
3      20
9       5
Name: count, dtype: int64

In [3]:
## we are creating a neural network which takes 11 features as input and predicts the quality of the wine as output.

## Split the data into training, validation, and testing sets

train, test = train_test_split(data, test_size=0.2, random_state=42)  



In [4]:
train["quality"].values

array([6, 5, 6, ..., 6, 6, 8], shape=(3918,))

In [29]:
# quality is the target variable and the rest are the features

X_train = train.drop(columns=["quality"])
y_train = train["quality"].values

## spliting the training data into training and validation sets

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)


## test set
X_test = test.drop(columns=["quality"])
y_test = test["quality"].values


# features
X_train = X_train.to_numpy().astype(np.float32)
X_val = X_val.to_numpy().astype(np.float32)
X_test = X_test.to_numpy().astype(np.float32)

# targets
y_train = y_train.astype(np.float32)
y_val = y_val.astype(np.float32)
y_test = y_test.astype(np.float32)

# mlflow.tensorflow.autolog(log_models=False)


In [30]:
## ANN model

def train_model(params, epochs, X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val, X_test=X_test, y_test=y_test):

    ## Define the model architecture 

    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)

    model = keras.Sequential([
        keras.Input(shape=(X_train.shape[1],), dtype=np.float32),
        keras.layers.Normalization(mean=mean, variance=std**2),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(1)
    ])

    ## Compile the model

    optimizer = keras.optimizers.SGD(
        learning_rate=params["lr"],
        momentum=params["momentum"],
        nesterov=True
    )

    model.compile(optimizer=optimizer, loss="mean_squared_error", 
    metrics=[keras.metrics.RootMeanSquaredError()])

    

    ## Train the model with lr and momentum as hyperparameters with MLflow autologging

    with mlflow.start_run(nested=True):
        model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=epochs, batch_size=8,verbose=0)

        ## Evaluate the model on the test set
        eval_result = model.evaluate(X_val, y_val, batch_size=8,verbose=0) 

        eval_rmse = eval_result[1]

        ## Log the hyperparameters and the evaluation metric to MLflow
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse", eval_rmse)

        ## log the model 
        # mlflow.tensorflow.log_model(model, "model", signature=signature)

        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}




In [31]:
## next we will use hyperopt to find the best hyperparameters for our model

def objective(params):
    result = train_model(
        params, 
        epochs=3, 
        X_train=X_train, 
        y_train=y_train, 
        X_val=X_val, 
        y_val=y_val, 
        X_test=X_test, 
        y_test=y_test)
    return result



In [32]:
space = {
    "lr": hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum": hp.uniform("momentum", 0.0, 1.0)
}



In [34]:
mlflow.set_tracking_uri("http://localhost:5000")

mlflow.set_experiment("wine_quality_prediction")
#comduct hyperparameter tuning with hyperopt and log the results to MLflow

with mlflow.start_run():
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials
    )

    # fetch the best model from the trials
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # infer the signature of the model for logging to MLflow
    signature = infer_signature(X_train, best_run["model"].predict(X_train))

    # log the best parameters and the best metric to MLflow
    mlflow.log_params(best)
    mlflow.log_metric("best_eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "best_model", signature=signature) 

# print out the best hyperparameters and the best evaluation metric
print("Best Hyperparameters:", best)
print("Best Evaluation RMSE:", best_run["loss"])



  0%|          | 0/4 [00:00<?, ?trial/s, best loss=?]

🏃 View run upbeat-eel-112 at: http://localhost:5000/#/experiments/1/runs/880394b99e884d0cb93f22da9871c7ae

🧪 View experiment at: http://localhost:5000/#/experiments/1

 25%|██▌       | 1/4 [00:05<00:16,  5.41s/trial, best loss: 4.030059337615967]

🏃 View run casual-snake-558 at: http://localhost:5000/#/experiments/1/runs/40253a8b9abb449fa6d93d2bba1b4a17

🧪 View experiment at: http://localhost:5000/#/experiments/1                  

 50%|█████     | 2/4 [00:08<00:08,  4.25s/trial, best loss: 1.0940043926239014]

🏃 View run incongruous-roo-501 at: http://localhost:5000/#/experiments/1/runs/f7a856b6543d49fa8d4366d336ffa602

🧪 View experiment at: http://localhost:5000/#/experiments/1                   

 75%|███████▌  | 3/4 [00:12<00:03,  3.85s/trial, best loss: 0.7129126787185669]

🏃 View run magnificent-boar-991 at: http://localhost:5000/#/experiments/1/runs/2d0c49b855d943a7882a8665bef7d1dc

🧪 View experiment at: http://localhost:5000/#/experiments/1                   

100%|██████████| 4/4 [00:17<00:00,  4.41s/trial, best loss: 0.7129126787185669]
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 905us/step


2026/05/22 21:14:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run amazing-sow-833 at: http://localhost:5000/#/experiments/1/runs/1f3aaf5c8ee14e6e9176896fb974ebce
🧪 View experiment at: http://localhost:5000/#/experiments/1
Best Hyperparameters: {'lr': np.float64(0.04682551294342236), 'momentum': np.float64(0.46040206339745826)}
Best Evaluation RMSE: 0.7129126787185669


In [36]:
## Inferencing the model with the best hyperparameters on the test set

from mlflow.models import validate_serving_input

model_uri = f"runs:/1f3aaf5c8ee14e6e9176896fb974ebce/best_model"

from mlflow.models import convert_input_example_to_serving_input

serving_payload = convert_input_example_to_serving_input(X_test)

validate_serving_input(model_uri, serving_payload)



31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


array([[6.1866145],
       [6.5350413],
       [6.5040846],
       [5.783627 ],
       [6.2938776],
       [6.67548  ],
       [5.1736813],
       [5.097127 ],
       [5.8399224],
       [5.2535515],
       [6.5406947],
       [4.666721 ],
       [6.5366597],
       [5.440238 ],
       [6.2968225],
       [5.5626073],
       [6.7687073],
       [5.953799 ],
       [6.1919613],
       [5.602919 ],
       [5.35131  ],
       [5.479269 ],
       [5.4614854],
       [6.229534 ],
       [5.868757 ],
       [5.5735846],
       [5.4817724],
       [6.249335 ],
       [5.990772 ],
       [5.3308244],
       [5.4819746],
       [5.666113 ],
       [5.56525  ],
       [5.503632 ],
       [5.704819 ],
       [6.510587 ],
       [6.136654 ],
       [5.432579 ],
       [5.67649  ],
       [5.8483367],
       [5.547843 ],
       [5.509442 ],
       [5.973281 ],
       [5.4819374],
       [5.045436 ],
       [5.7145133],
       [5.823629 ],
       [5.2862115],
       [5.667075 ],
       [5.4517155],


In [43]:
model_uri = f"runs:/1f3aaf5c8ee14e6e9176896fb974ebce/best_model"
y_pred = mlflow.pyfunc.load_model(model_uri).predict(X_test)

df = pd.DataFrame(
    {
        "y_true": np.array(y_test).flatten(),
        "y_pred": np.array(y_pred).flatten()
    }
)

print(df.head())

 1/31 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step

31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
   y_true    y_pred
0     7.0  6.186615
1     8.0  6.535041
2     8.0  6.504085
3     5.0  5.783627
4     7.0  6.293878


In [33]:
mlflow.end_run()

# 🍷 Wine Quality Prediction ML Project

## 📌 Project Overview

This project predicts the **quality of wine** based on physicochemical properties using Machine Learning.

- Target variable: `quality` (range: 3–9)
- Type: **Regression problem**
- Tools used: Python, Scikit-learn, MLflow, Hyperopt

---

## 📊 Dataset Overview

Each wine sample contains chemical features:

- fixed acidity
- volatile acidity
- citric acid
- residual sugar
- chlorides
- free sulfur dioxide
- total sulfur dioxide
- density
- pH
- sulphates
- alcohol

Target:
```text
quality score (3 to 9)

6    2198
5    1457
7     880
8     175
4     163
3      20
9       5

📌 Key Insight:
Highly imbalanced dataset
Majority of samples are in 5–6–7 range
Extreme values (3, 8, 9) are rare


🧠 Key Problem: Imbalanced Regression Target

Because of imbalance:

Model becomes biased toward average value (~6)
Poor performance on rare high-quality wines
Classic regression-to-the-mean effect


📉 Model Behavior
Example Predictions:
Actual	Predicted
7.0	6.18
8.0	6.53
8.0	6.50
5.0	5.78
Insight:
Model underestimates high-quality wines, Predictions cluster around ~6
⚙️ MLflow + Hyperopt Pipeline
Hyperopt used for hyperparameter tuning, MLflow used for experiment tracking
Best model selected based on RMSE

❗ Common Issue Faced
Flattening Error:
ValueError: Per-column arrays must each be 1-dimensional
Fix:
np.array(y_test).flatten()
np.array(y_pred).flatten()


📊 Model Evaluation
Metric Used:
RMSE = 0.7129
Interpretation:
RMSE Range	Meaning
< 0.5	Excellent
0.5–0.8	Good
0.8–1.0	Average
> 1.0	Weak
Result:
Model performance: Good
But biased toward mean predictions